In [1]:
import cv2
import numpy as np
import joblib
from ultralytics import YOLO

#loading models
pose_model = YOLO('yolov8n-pose.pt')
classifier = joblib.load('model.pkl')

#keypoints to drop same as in training
drop_kps = [13,14,15,16]
drop_indices = []
for i in drop_kps:
    drop_indices += [i*3, i*3+1, i*3+2]

print("Models loaded.")
print(f"Dropping feature indices: {drop_indices}")


Models loaded.
Dropping feature indices: [39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50]


In [2]:
cap = cv2.VideoCapture(0)
print("Press 'q' to quit")

while cap.isOpened():
    ret,frame = cap.read()
    if not ret:
        break

    results = pose_model(frame, verbose=False)

    if results[0].keypoints is not None and len(results[0].keypoints.data) > 0:
        kps = results[0].keypoints.data[0].cpu().numpy().flatten()

        features = np.delete(kps, drop_indices).reshape(1,-1)
        prediction = classifier.predict(features)[0]

        label = "GOOD POSTURE" if prediction == 0 else "SLOUCH"
        color = (0,255,0) if prediction == 0 else (0,0,255)

        cv2.putText(frame, label, (30,60), cv2.FONT_HERSHEY_SIMPLEX, 1.5, color, 3)

    annotated = results[0].plot()
    cv2.putText(annotated, label if results[0].keypoints is not None else "No person detected", (30,60), cv2.FONT_HERSHEY_SIMPLEX, 1.5, color if results[0].keypoints is not None else (255,255,255), 3)

    cv2.imshow('Posture Monitor', annotated)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


Press 'q' to quit


C:\Users\DELL\Desktop\Machine Learning\Projects\posture-monitor\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MLPClassifier was fitted with feature names
  warnings.warn(


error: OpenCV(4.13.0) :-1: error: (-5:Bad argument) in function 'putText'
> Overload resolution failed:
>  - putText() missing required argument 'org' (pos 3)
>  - putText() missing required argument 'org' (pos 3)


In [ ]:
q